In [2]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport
import pickle
import math

from sklearn.preprocessing import LabelEncoder

%matplotlib inline
pd.set_option('display.max_columns', None)
import warnings
warnings.filterwarnings("ignore")

c:\Users\ahmed\Desktop\Machine Learning Projects\End to End ML Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv(r'data\stud.csv')
df.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [5]:
# Create a data report and save it as an HTML file
profile = ProfileReport(df, title="My Data Report", explorative=True)
profile.to_file("report.html")

Export report to file: 100%|██████████| 1/1 [00:00<00:00, 180.71it/s]


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   gender                       1000 non-null   object
 1   race_ethnicity               1000 non-null   object
 2   parental_level_of_education  1000 non-null   object
 3   lunch                        1000 non-null   object
 4   test_preparation_course      1000 non-null   object
 5   math_score                   1000 non-null   int64 
 6   reading_score                1000 non-null   int64 
 7   writing_score                1000 non-null   int64 
dtypes: int64(3), object(5)
memory usage: 62.6+ KB


In [7]:
# describe numerical columns
df.select_dtypes(include = 'number').describe()

,math_score,reading_score,writing_score
count,1000.00000,1000.000000,1000.000000
mean,66.08900,69.169000,68.054000
std,15.16308,14.600192,15.195657
min,0.00000,17.000000,10.000000
25%,57.00000,59.000000,57.750000
50%,66.00000,70.000000,69.000000
75%,77.00000,79.000000,79.000000
max,100.00000,100.000000,100.000000


In [8]:
# describe numerical columns
df.select_dtypes(include = 'O').describe()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course
count,1000,1000,1000,1000,1000
unique,2,5,6,2,2
top,female,group C,some college,standard,none
freq,518,319,226,645,642


In [ ]:
# Check if there are null values
df.isnull().sum()

gender                         0
race_ethnicity                 0
parental_level_of_education    0
lunch                          0
test_preparation_course        0
math_score                     0
reading_score                  0
writing_score                  0
dtype: int64

In [ ]:
# Check if there are duplicated rows
df.duplicated().sum()

np.int64(0)

In [29]:
def explore_dataframe(df):
    feature_summary = []

    for col in df.columns:
        col_data = df[col]
        summary = {
            'Feature': col,
            'Data Type': col_data.dtype,
            'Num Missing': col_data.isnull().sum(),
            'Num Unique': col_data.nunique(),
            'Unique Values (first 10)': col_data.unique()[:10],
            'Most Frequent Value': col_data.mode().iloc[0] if not col_data.mode().empty else None,
            'Frequency of Most Frequent': col_data.value_counts().iloc[0] if not col_data.value_counts().empty else None,
        }

        feature_summary.append(summary)

    summary_df = pd.DataFrame(feature_summary)
    return summary_df

# Example usage:
# df = pd.read_csv('your_file.csv')
summary = explore_dataframe(df)
pd.set_option('display.max_columns', None)  # Show all columns
summary

,Feature,Data Type,Num Missing,Num Unique,Unique Values (first 10),Most Frequent Value,Frequency of Most Frequent
0,gender,object,0,2,"[female, male]",female,518
1,race_ethnicity,object,0,5,"[group B, group C, group A, group D, group E]",group C,319
2,parental_level_of_education,object,0,6,"[bachelor's degree, some college, master's deg...",some college,226
3,lunch,object,0,2,"[standard, free/reduced]",standard,645
4,test_preparation_course,object,0,2,"[none, completed]",none,642
5,math_score,int64,0,81,"[72, 69, 90, 47, 76, 71, 88, 40, 64, 38]",65,36
6,reading_score,int64,0,72,"[72, 90, 95, 57, 78, 83, 43, 64, 60, 54]",72,34
7,writing_score,int64,0,77,"[74, 88, 93, 44, 75, 78, 92, 39, 67, 50]",74,35


In [37]:
numerical_features = df.select_dtypes(include = 'number').columns
categorical_features = df.select_dtypes(include = 'object').columns

# discrete features
discrete_features = [feature for feature in df if len(df[feature].unique()) <= 17]

# Continuous features
continuous_features = [feature for feature in df if feature not in discrete_features]

print('Number of Numerical features: ', len(numerical_features), )
print('Number of categorical features: ', len(categorical_features))
print('Number of discrete features: ', len(discrete_features))
print('Number of continuous features: ', len(continuous_features))


Number of Numerical features:  3
Number of categorical features:  5
Number of discrete features:  5
Number of continuous features:  3


In [38]:
df['Total Score'] = df['math_score'] + df['writing_score'] + df['reading_score']